# Changing the mesh layout

The mesh layout controls the topology of the `m2m` (mesh-to-mesh) component of the graph.
By default, `weather-model-graphs` uses a **rectilinear** mesh, where nodes sit on a regular
rectangular grid and edges connect each node to its 8 nearest neighbours (cardinal + diagonal).

As of v0.4.0, a **triangular** mesh layout is also supported. This places nodes on an equilateral-
triangle lattice, giving each interior node exactly 6 neighbours instead of 8. The 6-connectivity
is more isotropic and can improve message-passing in graph neural network weather models.

In this notebook we use the [Keisler 2022](https://arxiv.org/abs/2202.07575) graph archetype to
contrast three variants:

1. **Default rectilinear** mesh (the archetype's built-in default)
2. **Rectilinear with finer mesh spacing** (more mesh nodes, denser connectivity)
3. **Triangular mesh** at the same spacing as variant 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import weather_model_graphs as wmg

## Set up a fake grid

We start from a regular 32 × 32 grid of Cartesian (x, y) coordinates. These represent the
locations of the input/output data (grid nodes in g2m / m2g).

In [ ]:
xs, ys = np.meshgrid(np.linspace(0, 10, 32), np.linspace(0, 10, 32))
xy = np.stack([xs.flatten(), ys.flatten()], axis=-1)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(xy[:, 0], xy[:, 1], s=2)
ax.set_aspect(1)
ax.set_title("Grid nodes")

## Example 1 — Rectilinear mesh (default spacing)

`create_keisler_graph` uses `mesh_layout='rectilinear'` with `mesh_node_distance=3` by default.
Each interior mesh node connects to its 8 neighbours (4-star cardinal + 4 diagonals).

In [ ]:
graph_rectilinear = wmg.create.archetype.create_keisler_graph(
    coords=xy,
    mesh_node_distance=3,
)

m2m_rectilinear = wmg.split_graph_by_edge_attribute(graph_rectilinear, attr="component")[
    "m2m"
]

print(f"Mesh nodes : {m2m_rectilinear.number_of_nodes()}")
print(f"Mesh edges : {m2m_rectilinear.number_of_edges()}")

fig, ax = plt.subplots(figsize=(5, 5))
wmg.visualise.nx_draw_with_pos_and_attr(m2m_rectilinear, ax=ax, node_size=30)
ax.set_title("Rectilinear mesh — default spacing (mesh_node_distance=3)")

## Example 2 — Rectilinear mesh with finer spacing

Halving `mesh_node_distance` roughly quadruples the number of mesh nodes and gives a denser
rectilinear mesh over the same domain.

In [ ]:
graph_fine = wmg.create.archetype.create_keisler_graph(
    coords=xy,
    mesh_node_distance=1.5,
)

m2m_fine = wmg.split_graph_by_edge_attribute(graph_fine, attr="component")["m2m"]

print(f"Mesh nodes : {m2m_fine.number_of_nodes()}")
print(f"Mesh edges : {m2m_fine.number_of_edges()}")

fig, ax = plt.subplots(figsize=(5, 5))
wmg.visualise.nx_draw_with_pos_and_attr(m2m_fine, ax=ax, node_size=10)
ax.set_title("Rectilinear mesh — finer spacing (mesh_node_distance=1.5)")

## Example 3 — Triangular mesh

Setting `mesh_layout='triangular'` places nodes on an equilateral-triangle lattice.
Each interior node has exactly **6 neighbours** (vs. 8 for rectilinear), which provides
more isotropic spatial connectivity.

We keep the same g2m / m2g connectivity settings as the Keisler archetype (within-radius
encoding, 4-nearest-neighbour decoding) and use the same mesh spacing as Example 1.

In [ ]:
graph_triangular = wmg.create.create_all_graph_components(
    coords=xy,
    mesh_layout="triangular",
    mesh_layout_kwargs=dict(mesh_node_spacing=3),
    m2m_connectivity="flat",
    g2m_connectivity="within_radius",
    g2m_connectivity_kwargs=dict(rel_max_dist=0.51),
    m2g_connectivity="nearest_neighbours",
    m2g_connectivity_kwargs=dict(max_num_neighbours=4),
)

m2m_triangular = wmg.split_graph_by_edge_attribute(graph_triangular, attr="component")[
    "m2m"
]

print(f"Mesh nodes : {m2m_triangular.number_of_nodes()}")
print(f"Mesh edges : {m2m_triangular.number_of_edges()}")

fig, ax = plt.subplots(figsize=(5, 5))
wmg.visualise.nx_draw_with_pos_and_attr(m2m_triangular, ax=ax, node_size=30)
ax.set_title("Triangular mesh (mesh_node_spacing=3)")

## Side-by-side comparison

Plotting the `m2m` component of all three graphs side by side makes the difference in
topology clear: rectilinear nodes form a square grid with 8-connectivity, while triangular
nodes form a hexagonal-offset grid with 6-connectivity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    (m2m_rectilinear, "Rectilinear\n(default spacing)", 30),
    (m2m_fine, "Rectilinear\n(finer spacing)", 10),
    (m2m_triangular, "Triangular\n(same spacing as default)", 30),
]

for ax, (graph, title, ns) in zip(axes, configs):
    wmg.visualise.nx_draw_with_pos_and_attr(graph, ax=ax, node_size=ns)
    ax.set_title(title)

fig.tight_layout()